In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [3]:
# Install required libraries if not already installed in your environment
# !pip install geopandas folium shapely pandas numpy matplotlib

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, Point
import folium
import branca.colormap as cm

print("Libraries imported successfully. Ready for spatial analysis.")

Libraries imported successfully. Ready for spatial analysis.


In [4]:
# Step 1: Define the bounding box for Berlin (approximate coordinates)
berlin_bbox = {
    'min_lon': 13.1,
    'max_lon': 13.7,
    'min_lat': 52.35,
    'max_lat': 52.65
}

# Step 2: Create a grid of polygons (districts/zones) across Berlin
grid_size = 0.05  # Size of each grid cell in degrees
polygons = []
for lon in np.arange(berlin_bbox['min_lon'], berlin_bbox['max_lon'], grid_size):
    for lat in np.arange(berlin_bbox['min_lat'], berlin_bbox['max_lat'], grid_size):
        polygons.append(Polygon([
            (lon, lat),
            (lon + grid_size, lat),
            (lon + grid_size, lat + grid_size),
            (lon, lat + grid_size)
        ]))

# Create a GeoDataFrame representing the spatial grid
gdf_berlin = gpd.GeoDataFrame({'geometry': polygons}, crs="EPSG:4326")

# Step 3: Generate synthetic parameters for each grid cell
np.random.seed(42)
num_cells = len(gdf_berlin)

# Solar Potential (MWh/year) - Higher is better
gdf_berlin['solar_potential'] = np.random.uniform(500, 2000, num_cells)

# Land Price (EUR/sqm) - Lower is better for project economics
gdf_berlin['land_price'] = np.random.uniform(100, 1500, num_cells)

# Grid Congestion / Peak Load Risk (0 to 1) - Higher means more need for peak shaving
gdf_berlin['grid_congestion'] = np.random.uniform(0.1, 0.95, num_cells)

print(f"Generated {num_cells} spatial zones for Berlin distribution grid analysis.")
gdf_berlin.head()

Generated 72 spatial zones for Berlin distribution grid analysis.


,geometry,solar_potential,land_price,grid_congestion
0,"POLYGON ((13.1 52.35, 13.15 52.35, 13.15 52.4,...",1061.810178,107.730964,0.342114
1,"POLYGON ((13.1 52.4, 13.15 52.4, 13.15 52.45, ...",1926.071460,1241.646000,0.131354
2,"POLYGON ((13.1 52.45, 13.15 52.45, 13.15 52.5,...",1597.990913,1089.600281,0.618130
3,"POLYGON ((13.1 52.5, 13.15 52.5, 13.15 52.55, ...",1397.987726,1120.610035,0.527277
4,"POLYGON ((13.1 52.55, 13.15 52.55, 13.15 52.6,...",734.027961,1179.778485,0.143757


In [5]:
# Step 4: Normalize data (Min-Max Scaling) to bring all metrics to a 0-1 scale
def normalize(series, inverse=False):
    if inverse:
        # For land price, lower is better, so we invert the scale
        return (series.max() - series) / (series.max() - series.min())
    return (series - series.min()) / (series.max() - series.min())

gdf_berlin['norm_solar'] = normalize(gdf_berlin['solar_potential'])
gdf_berlin['norm_land'] = normalize(gdf_berlin['land_price'], inverse=True)
gdf_berlin['norm_grid'] = normalize(gdf_berlin['grid_congestion'])

# Step 5: Calculate Final BESS Suitability Score
# Weights: Grid congestion (50%), Land price (30%), Solar potential (20%)
w_grid, w_land, w_solar = 0.5, 0.3, 0.2

gdf_berlin['suitability_score'] = (
    (gdf_berlin['norm_grid'] * w_grid) +
    (gdf_berlin['norm_land'] * w_land) +
    (gdf_berlin['norm_solar'] * w_solar)
)

# Sort to find the absolute best locations for the 5 MW BESS
gdf_berlin = gdf_berlin.sort_values(by='suitability_score', ascending=False)
best_location = gdf_berlin.iloc[0]

print(f"Top Suitability Score: {best_location['suitability_score']:.2f}")

Top Suitability Score: 0.84


In [9]:
# Install required libraries if not already installed in your environment
# !pip install geopandas folium shapely pandas numpy matplotlib branca

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, Point
import folium
import branca.colormap as cm

print("Libraries imported successfully. Starting spatial analysis...")

# ==========================================
# STEP 1: Generate Spatial Grid for Berlin
# ==========================================
# Define the bounding box for Berlin (approximate coordinates)
berlin_bbox = {
    'min_lon': 13.1,
    'max_lon': 13.7,
    'min_lat': 52.35,
    'max_lat': 52.65
}

# Create a grid of polygons (districts/zones) across Berlin
grid_size = 0.05  # Size of each grid cell in degrees
polygons = []
for lon in np.arange(berlin_bbox['min_lon'], berlin_bbox['max_lon'], grid_size):
    for lat in np.arange(berlin_bbox['min_lat'], berlin_bbox['max_lat'], grid_size):
        polygons.append(Polygon([
            (lon, lat),
            (lon + grid_size, lat),
            (lon + grid_size, lat + grid_size),
            (lon, lat + grid_size)
        ]))

# Create a GeoDataFrame representing the spatial grid
gdf_berlin = gpd.GeoDataFrame({'geometry': polygons}, crs="EPSG:4326")

# ==========================================
# STEP 2: Generate Synthetic Data
# ==========================================
np.random.seed(42)
num_cells = len(gdf_berlin)

# Solar Potential (MWh/year) - Higher is better
gdf_berlin['solar_potential'] = np.random.uniform(500, 2000, num_cells)

# Land Price (EUR/sqm) - Lower is better for project economics
gdf_berlin['land_price'] = np.random.uniform(100, 1500, num_cells)

# Grid Congestion / Peak Load Risk (0 to 1) - Higher means more need for peak shaving
gdf_berlin['grid_congestion'] = np.random.uniform(0.1, 0.95, num_cells)

print(f"Generated {num_cells} spatial zones for Berlin distribution grid analysis.")

# ==========================================
# STEP 3: Multi-Criteria Decision Analysis (MCDA)
# ==========================================
# Normalize data (Min-Max Scaling) to bring all metrics to a 0-1 scale
def normalize(series, inverse=False):
    if inverse:
        # For land price, lower is better, so we invert the scale
        return (series.max() - series) / (series.max() - series.min())
    return (series - series.min()) / (series.max() - series.min())

gdf_berlin['norm_solar'] = normalize(gdf_berlin['solar_potential'])
gdf_berlin['norm_land'] = normalize(gdf_berlin['land_price'], inverse=True)
gdf_berlin['norm_grid'] = normalize(gdf_berlin['grid_congestion'])

# Calculate Final BESS Suitability Score
# Weights: Grid congestion (50%), Land price (30%), Solar potential (20%)
w_grid, w_land, w_solar = 0.5, 0.3, 0.2

gdf_berlin['suitability_score'] = (
    (gdf_berlin['norm_grid'] * w_grid) +
    (gdf_berlin['norm_land'] * w_land) +
    (gdf_berlin['norm_solar'] * w_solar)
)

# Sort to find the absolute best locations for the 5 MW BESS
gdf_berlin = gdf_berlin.sort_values(by='suitability_score', ascending=False)
best_location = gdf_berlin.iloc[0]

print(f"Top Suitability Score found: {best_location['suitability_score']:.2f}")

# ==========================================
# STEP 4: Interactive Visualization with Folium
# ==========================================
# Center the map on Berlin using reliable Esri tiles to avoid API/403 errors
berlin_map = folium.Map(
    location=[52.5200, 13.4050],
    zoom_start=11,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

# Define a color map for the suitability score
colormap = cm.LinearColormap(
    colors=['red', 'yellow', 'green'],
    vmin=gdf_berlin['suitability_score'].min(),
    vmax=gdf_berlin['suitability_score'].max(),
    caption='BESS Placement Suitability Score'
)
colormap.add_to(berlin_map)

# Add the grid zones to the map
for _, row in gdf_berlin.iterrows():
    sim_geo = gpd.GeoSeries(row['geometry'])
    geo_json = sim_geo.to_json()

    score = row['suitability_score']
    color = colormap(score)

    popup_text = (
        f"<b>Suitability Score:</b> {score:.2f}<br>"
        f"<b>Grid Congestion:</b> {row['grid_congestion']:.2f}<br>"
        f"<b>Land Price:</b> {row['land_price']:.0f} EUR/m²<br>"
        f"<b>Solar Potential:</b> {row['solar_potential']:.0f} MWh"
    )

    folium.GeoJson(
        geo_json,
        style_function=lambda feature, color=color: {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.6
        },
        tooltip=popup_text
    ).add_to(berlin_map)

# Highlight the absolute best location for the 5MW BESS
best_centroid = best_location['geometry'].centroid
folium.Marker(
    location=[best_centroid.y, best_centroid.x],
    popup="🏆 Optimal 5MW BESS Location",
    icon=folium.Icon(color='blue', icon='star')
).add_to(berlin_map)

# Display the map
berlin_map

Libraries imported successfully. Starting spatial analysis...
Generated 72 spatial zones for Berlin distribution grid analysis.
Top Suitability Score found: 0.84


In [10]:
import json
import base64
import os

def extract_images_from_notebook(notebook_path="notebooks/1_berlin_bess_spatial_analysis.ipynb", output_dir="outputs"):
    # ایجاد پوشه خروجی در صورت عدم وجود
    os.makedirs(output_dir, exist_ok=True)

    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
    except FileNotFoundError:
        print(f"❌ فایل {notebook_path} پیدا نشد. لطفاً مسیر فایل ژوپیتر را اصلاح کنید.")
        return

    image_count = 0

    # جستجو در تمام سلول‌های نوت‌بوک
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') == 'code':
            for output in cell.get('outputs', []):
                # بررسی وجود تصویر استاتیک در خروجی سلول
                if 'data' in output and 'image/png' in output['data']:
                    image_data = output['data']['image/png']

                    # هندل کردن فرمت‌های مختلف ذخیره‌سازی base64 در ژوپیتر
                    if isinstance(image_data, list):
                        image_data = "".join(image_data)

                    image_bytes = base64.b64decode(image_data)
                    image_count += 1

                    image_path = os.path.join(output_dir, f"plot_output_{image_count}.png")
                    with open(image_path, 'wb') as img_file:
                        img_file.write(image_bytes)
                    print(f"✅ تصویر استخراج و ذخیره شد: {image_path}")

    if image_count == 0:
        print("⚠️ هیچ تصویر استاتیکی (PNG) در فایل ژوپیتر پیدا نشد.")
        print("💡 نکته: نقشه‌های Folium به صورت HTML تعاملی هستند و عکس استاتیک محسوب نمی‌شوند.")
        print("برای ذخیره نقشه Folium، کد berlin_map.save('outputs/map.html') را در ژوپیتر اجرا کنید.")

if __name__ == "__main__":
    # اگر فایل ژوپیتر شما نام یا مسیر دیگری دارد، آن را در خط زیر تغییر دهید
    extract_images_from_notebook(notebook_path="1_berlin_bess_spatial_analysis.ipynb")

❌ فایل 1_berlin_bess_spatial_analysis.ipynb پیدا نشد. لطفاً مسیر فایل ژوپیتر را اصلاح کنید.


In [11]:
berlin_map.save('/content/berlin_bess_map.html')

In [14]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt

# ۱. ساخت پوشه خروجی
os.makedirs('outputs', exist_ok=True)

# ۲. پیدا کردن هوشمند مسیر فایل
file_path = "data/berlin_grid_data.geojson"
if not os.path.exists(file_path):
    file_path = "../data/berlin_grid_data.geojson"

if not os.path.exists(file_path):
    print("❌ فایل دیتابیس پیدا نشد! اگر در گوگل کولب هستید، باید ابتدا فایل berlin_grid_data.geojson را آپلود کنید.")
else:
    # بارگذاری داده‌ها
    gdf = gpd.read_file(file_path)

    # ۳. محاسبه فرمول‌ها
    gdf['norm_solar'] = (gdf['solar_potential'] - gdf['solar_potential'].min()) / (gdf['solar_potential'].max() - gdf['solar_potential'].min())
    gdf['norm_land'] = (gdf['land_price'].max() - gdf['land_price']) / (gdf['land_price'].max() - gdf['land_price'].min()) # معکوس
    gdf['norm_grid'] = (gdf['grid_congestion'] - gdf['grid_congestion'].min()) / (gdf['grid_congestion'].max() - gdf['grid_congestion'].min())

    gdf['suitability_score'] = (gdf['norm_grid'] * 0.5) + (gdf['norm_land'] * 0.3) + (gdf['norm_solar'] * 0.2)

    # ۴. رسم نقشه
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    gdf.plot(
        column='suitability_score',
        cmap='RdYlGn',
        legend=True,
        ax=ax,
        edgecolor='black',
        linewidth=0.5,
        legend_kwds={'label': "Suitability Score"}
    )

    plt.title("Berlin BESS Optimal Placement", fontsize=16)
    plt.axis('off')

    # ۵. ذخیره فایل
    output_path = 'outputs/map_preview.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✅ تصویر نقشه با موفقیت ساخته شد: {output_path}")

❌ فایل دیتابیس پیدا نشد! اگر در گوگل کولب هستید، باید ابتدا فایل berlin_grid_data.geojson را آپلود کنید.
